In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
from typing import Optional, Tuple
import matplotlib.pyplot as plt
import numpy as np



In [ ]:

# ============================================================================
# 1. BASIC MULTI-HEAD LATENT ATTENTION
# ============================================================================

class MultiHeadLatentAttention(nn.Module):
    """
    Multi-Head Latent Attention (MLA) with low-rank compression.
    
    Key innovation: Compress K,V into low-dimensional latent space,
    reducing KV cache size dramatically while maintaining quality.
    """
    
    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        latent_dim: int,
        query_latent_dim: Optional[int] = None,
        dropout: float = 0.1
    ):
        """
        Args:
            embed_dim: Model dimension
            num_heads: Number of attention heads
            latent_dim: Dimension of compressed KV space
            query_latent_dim: Dimension of compressed Q space (default: latent_dim * 3)
            dropout: Dropout probability
        """
        super(MultiHeadLatentAttention, self).__init__()
        
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.latent_dim = latent_dim
        self.query_latent_dim = query_latent_dim or (latent_dim * 3)
        
        # Compression layers
        self.q_compress = nn.Linear(embed_dim, self.query_latent_dim, bias=False)
        self.kv_compress = nn.Linear(embed_dim, latent_dim, bias=False)
        
        # Decompression layers (per-head)
        self.q_decompress = nn.ModuleList([
            nn.Linear(self.query_latent_dim, self.head_dim, bias=False)
            for _ in range(num_heads)
        ])
        self.k_decompress = nn.ModuleList([
            nn.Linear(latent_dim, self.head_dim, bias=False)
            for _ in range(num_heads)
        ])
        self.v_decompress = nn.ModuleList([
            nn.Linear(latent_dim, self.head_dim, bias=False)
            for _ in range(num_heads)
        ])
        
        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights with small random values."""
        nn.init.normal_(self.q_compress.weight, std=0.02)
        nn.init.normal_(self.kv_compress.weight, std=0.02)
        
        for i in range(self.num_heads):
            nn.init.normal_(self.q_decompress[i].weight, std=0.02 / math.sqrt(self.query_latent_dim))
            nn.init.normal_(self.k_decompress[i].weight, std=0.02 / math.sqrt(self.latent_dim))
            nn.init.normal_(self.v_decompress[i].weight, std=0.02 / math.sqrt(self.latent_dim))
        
        nn.init.normal_(self.out_proj.weight, std=0.02)
        if self.out_proj.bias is not None:
            nn.init.zeros_(self.out_proj.bias)
    
    def forward(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
        return_attention: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args:
            x: Input tensor [batch, seq_len, embed_dim]
            mask: Optional attention mask
            return_attention: Whether to return attention weights
        
        Returns:
            output: [batch, seq_len, embed_dim]
            attention_weights: [batch, num_heads, seq_len, seq_len] (optional)
        """
        batch_size, seq_len, embed_dim = x.size()
        
        # Step 1: Compress to latent space
        c_q = self.q_compress(x)      # [batch, seq_len, query_latent_dim]
        c_kv = self.kv_compress(x)    # [batch, seq_len, latent_dim]
        
        # Step 2: Decompress per head
        all_q = []
        all_k = []
        all_v = []
        
        for i in range(self.num_heads):
            q_i = self.q_decompress[i](c_q)  # [batch, seq_len, head_dim]
            k_i = self.k_decompress[i](c_kv)
            v_i = self.v_decompress[i](c_kv)
            
            all_q.append(q_i)
            all_k.append(k_i)
            all_v.append(v_i)
        
        # Stack: [batch, num_heads, seq_len, head_dim]
        Q = torch.stack(all_q, dim=1)
        K = torch.stack(all_k, dim=1)
        V = torch.stack(all_v, dim=1)
        
        # Step 3: Compute attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Step 4: Apply attention to values
        attn_output = torch.matmul(attn_weights, V)
        
        # Step 5: Reshape and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.embed_dim
        )
        
        output = self.out_proj(attn_output)
        
        if return_attention:
            return output, attn_weights
        return output, None


# ============================================================================
# 2. OPTIMIZED MLA WITH KV CACHE
# ============================================================================

class CachedMultiHeadLatentAttention(nn.Module):
    """
    MLA with KV caching for efficient autoregressive generation.
    
    Key advantage: Cache only the compressed latent representation,
    not full K,V for each head!
    """
    
    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        latent_dim: int,
        query_latent_dim: Optional[int] = None,
        dropout: float = 0.1
    ):
        super(CachedMultiHeadLatentAttention, self).__init__()
        
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.latent_dim = latent_dim
        self.query_latent_dim = query_latent_dim or (latent_dim * 3)
        
        # Compression
        self.q_compress = nn.Linear(embed_dim, self.query_latent_dim, bias=False)
        self.kv_compress = nn.Linear(embed_dim, latent_dim, bias=False)
        
        # Decompression (per-head)
        self.q_decompress = nn.ModuleList([
            nn.Linear(self.query_latent_dim, self.head_dim, bias=False)
            for _ in range(num_heads)
        ])
        self.k_decompress = nn.ModuleList([
            nn.Linear(latent_dim, self.head_dim, bias=False)
            for _ in range(num_heads)
        ])
        self.v_decompress = nn.ModuleList([
            nn.Linear(latent_dim, self.head_dim, bias=False)
            for _ in range(num_heads)
        ])
        
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights."""
        nn.init.normal_(self.q_compress.weight, std=0.02)
        nn.init.normal_(self.kv_compress.weight, std=0.02)
        
        for i in range(self.num_heads):
            nn.init.normal_(self.q_decompress[i].weight, std=0.02 / math.sqrt(self.query_latent_dim))
            nn.init.normal_(self.k_decompress[i].weight, std=0.02 / math.sqrt(self.latent_dim))
            nn.init.normal_(self.v_decompress[i].weight, std=0.02 / math.sqrt(self.latent_dim))
        
        nn.init.normal_(self.out_proj.weight, std=0.02)
    
    def forward(
        self,
        x: torch.Tensor,
        past_c_kv: Optional[torch.Tensor] = None,
        use_cache: bool = False,
        mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args:
            x: Input [batch, seq_len, embed_dim]
            past_c_kv: Cached compressed KV from previous steps
            use_cache: Whether to return updated cache
            mask: Optional attention mask
        
        Returns:
            output: [batch, seq_len, embed_dim]
            present_c_kv: Updated compressed KV cache (if use_cache=True)
        """
        batch_size, seq_len, _ = x.size()
        
        # Compress
        c_q = self.q_compress(x)
        c_kv = self.kv_compress(x)  # This is what we cache!
        
        # Concatenate with past if provided
        if past_c_kv is not None:
            c_kv = torch.cat([past_c_kv, c_kv], dim=1)
        
        present_c_kv = c_kv if use_cache else None
        
        total_seq_len = c_kv.size(1)
        
        # Decompress for all heads
        all_q = []
        all_k = []
        all_v = []
        
        for i in range(self.num_heads):
            q_i = self.q_decompress[i](c_q)
            k_i = self.k_decompress[i](c_kv)  # Use full cache
            v_i = self.v_decompress[i](c_kv)
            
            all_q.append(q_i)
            all_k.append(k_i)
            all_v.append(v_i)
        
        Q = torch.stack(all_q, dim=1)
        K = torch.stack(all_k, dim=1)
        V = torch.stack(all_v, dim=1)
        
        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, self.embed_dim
        )
        
        output = self.out_proj(attn_output)
        
        return output, present_c_kv


# ============================================================================
# 3. CONVERSION: STANDARD MHA TO MLA
# ============================================================================

class StandardMultiHeadAttention(nn.Module):
    """Standard MHA for comparison and conversion."""
    
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super(StandardMultiHeadAttention, self).__init__()
        
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()
        
        Q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        output = torch.matmul(attn, V)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        output = self.out_proj(output)
        
        return output


def convert_mha_to_mla_svd(
    mha: StandardMultiHeadAttention,
    latent_dim: int,
    query_latent_dim: Optional[int] = None
) -> MultiHeadLatentAttention:
    """
    Convert trained MHA to MLA using SVD for initialization.
    
    Args:
        mha: Trained StandardMultiHeadAttention module
        latent_dim: Target latent dimension for KV
        query_latent_dim: Target latent dimension for Q
    
    Returns:
        mla: Initialized MultiHeadLatentAttention
    """
    mla = MultiHeadLatentAttention(
        embed_dim=mha.embed_dim,
        num_heads=mha.num_heads,
        latent_dim=latent_dim,
        query_latent_dim=query_latent_dim
    )
    
    # Get K, V projection weights
    W_k = mha.k_proj.weight.data  # [embed_dim, embed_dim]
    W_v = mha.v_proj.weight.data
    
    # SVD on K projection
    U_k, S_k, Vt_k = torch.linalg.svd(W_k.T, full_matrices=False)
    
    # Take top latent_dim components
    W_kv_down = U_k[:, :latent_dim] @ torch.diag(torch.sqrt(S_k[:latent_dim]))
    W_k_up_base = torch.diag(torch.sqrt(S_k[:latent_dim])) @ Vt_k[:latent_dim, :]
    
    # Initialize compression
    mla.kv_compress.weight.data = W_kv_down.T
    
    # Split and assign to per-head decompression
    for i in range(mla.num_heads):
        start = i * mla.head_dim
        end = start + mla.head_dim
        
        # K decompression for this head
        mla.k_decompress[i].weight.data = W_k_up_base[:, start:end].T
    
    # Similar for V (simplified here)
    U_v, S_v, Vt_v = torch.linalg.svd(W_v.T, full_matrices=False)
    W_v_up_base = torch.diag(torch.sqrt(S_v[:latent_dim])) @ Vt_v[:latent_dim, :]
    
    for i in range(mla.num_heads):
        start = i * mla.head_dim
        end = start + mla.head_dim
        mla.v_decompress[i].weight.data = W_v_up_base[:, start:end].T
    
    # Copy Q projection (can use SVD or direct copy)
    W_q = mha.q_proj.weight.data
    U_q, S_q, Vt_q = torch.linalg.svd(W_q.T, full_matrices=False)
    
    q_latent = mla.query_latent_dim
    W_q_down = U_q[:, :q_latent] @ torch.diag(torch.sqrt(S_q[:q_latent]))
    W_q_up_base = torch.diag(torch.sqrt(S_q[:q_latent])) @ Vt_q[:q_latent, :]
    
    mla.q_compress.weight.data = W_q_down.T
    
    for i in range(mla.num_heads):
        start = i * mla.head_dim
        end = start + mla.head_dim
        mla.q_decompress[i].weight.data = W_q_up_base[:, start:end].T
    
    # Copy output projection
    mla.out_proj.weight.data = mha.out_proj.weight.data.clone()
    if mla.out_proj.bias is not None and mha.out_proj.bias is not None:
        mla.out_proj.bias.data = mha.out_proj.bias.data.clone()
    
    return mla


# ============================================================================
# 4. COMPLETE TRANSFORMER BLOCK WITH MLA
# ============================================================================

class MLATransformerBlock(nn.Module):
    """Complete transformer block using Multi-Head Latent Attention."""
    
    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        latent_dim: int,
        ff_dim: Optional[int] = None,
        dropout: float = 0.1
    ):
        super(MLATransformerBlock, self).__init__()
        
        if ff_dim is None:
            ff_dim = 4 * embed_dim
        
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        
        self.attn = MultiHeadLatentAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            latent_dim=latent_dim,
            dropout=dropout
        )
        
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x, mask=None):
        # Attention with residual
        attn_out, _ = self.attn(self.ln1(x), mask=mask)
        x = x + attn_out
        
        # FFN with residual
        x = x + self.ffn(self.ln2(x))
        
        return x


# ============================================================================
# 5. PERFORMANCE BENCHMARKING
# ============================================================================

class MLABenchmark:
    """Benchmark MLA vs MHA."""
    
    @staticmethod
    def calculate_kv_cache_size(
        seq_len: int,
        embed_dim: int,
        num_heads: int,
        latent_dim: int,
        num_layers: int = 32
    ):
        """Calculate KV cache sizes."""
        head_dim = embed_dim // num_heads
        
        # Standard MHA
        mha_cache = 2 * num_layers * num_heads * head_dim * seq_len * 2  # FP16
        
        # MLA
        mla_cache = num_layers * latent_dim * seq_len * 2  # FP16
        
        return {
            'mha_cache_mb': mha_cache / (1024**2),
            'mla_cache_mb': mla_cache / (1024**2),
            'reduction_factor': mha_cache / mla_cache,
            'memory_saved_mb': (mha_cache - mla_cache) / (1024**2)
        }
    
    @staticmethod
    def benchmark_speed(
        batch_size: int,
        seq_len: int,
        embed_dim: int,
        num_heads: int,
        latent_dim: int,
        num_iterations: int = 100
    ):
        """Benchmark inference speed."""
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        x = torch.randn(batch_size, seq_len, embed_dim, device=device)
        
        # MHA
        mha = StandardMultiHeadAttention(embed_dim, num_heads).to(device)
        mha.eval()
        
        # Warmup
        with torch.no_grad():
            for _ in range(10):
                _ = mha(x)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        start = time.time()
        with torch.no_grad():
            for _ in range(num_iterations):
                _ = mha(x)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        mha_time = (time.time() - start) / num_iterations
        
        # MLA
        mla = MultiHeadLatentAttention(embed_dim, num_heads, latent_dim).to(device)
        mla.eval()
        
        with torch.no_grad():
            for _ in range(10):
                _ = mla(x)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        start = time.time()
        with torch.no_grad():
            for _ in range(num_iterations):
                _ = mla(x)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        mla_time = (time.time() - start) / num_iterations
        
        return {
            'mha_time_ms': mha_time * 1000,
            'mla_time_ms': mla_time * 1000,
            'relative_speed': mha_time / mla_time
        }
    
    @staticmethod
    def count_parameters(model):
        """Count trainable parameters."""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================================
# 6. DEMONSTRATION EXAMPLES
# ============================================================================

def demo_basic_mla():
    """Demonstrate basic MLA usage."""
    print("="*80)
    print("DEMO 1: Basic Multi-Head Latent Attention")
    print("="*80)
    
    batch_size = 2
    seq_len = 10
    embed_dim = 256
    num_heads = 16
    latent_dim = 64  # Compression!
    
    # Create MLA
    mla = MultiHeadLatentAttention(embed_dim, num_heads, latent_dim)
    
    # Create input
    x = torch.randn(batch_size, seq_len, embed_dim)
    
    # Forward pass
    output, attn = mla(x, return_attention=True)
    
    print(f"\nConfiguration:")
    print(f"  Embed dim: {embed_dim}")
    print(f"  Num heads: {num_heads}")
    print(f"  Head dim: {embed_dim // num_heads}")
    print(f"  Latent dim: {latent_dim}")
    print(f"  Compression ratio: {(num_heads * (embed_dim // num_heads)) / latent_dim:.1f}x")
    
    print(f"\nShapes:")
    print(f"  Input: {x.shape}")
    print(f"  Output: {output.shape}")
    print(f"  Attention: {attn.shape}")
    
    # Parameter count
    mha_equiv_params = 4 * embed_dim * embed_dim
    mla_params = MLABenchmark.count_parameters(mla)
    
    print(f"\nParameters:")
    print(f"  MLA: {mla_params:,}")
    print(f"  MHA (equiv): {mha_equiv_params:,}")
    print(f"  Difference: {(1 - mla_params/mha_equiv_params)*100:.1f}% reduction")


def demo_kv_cache():
    """Demonstrate KV caching with MLA."""
    print("\n" + "="*80)
    print("DEMO 2: KV Caching with MLA")
    print("="*80)
    
    embed_dim = 128
    num_heads = 8
    latent_dim = 32
    
    mla = CachedMultiHeadLatentAttention(embed_dim, num_heads, latent_dim)
    mla.eval()
    
    print(f"\nConfiguration:")
    print(f"  Latent dim: {latent_dim}")
    print(f"  Full KV would be: {num_heads * (embed_dim // num_heads)} dims")
    print(f"  Compression: {(num_heads * (embed_dim // num_heads)) / latent_dim:.1f}x")
    
    print(f"\nSimulating generation:")
    
    # Initial token
    x = torch.randn(1, 1, embed_dim)
    output, cache = mla(x, use_cache=True)
    print(f"\nStep 1:")
    print(f"  Input: {x.shape}")
    print(f"  Cache: {cache.shape} (only {cache.numel()} values!)")
    
    # Generate more tokens
    for step in range(2, 5):
        x = torch.randn(1, 1, embed_dim)
        output, cache = mla(x, past_c_kv=cache, use_cache=True)
        print(f"\nStep {step}:")
        print(f"  Cache grows: {cache.shape}")
    
    total_cache = cache.numel()
    full_kv_cache = 2 * num_heads * (embed_dim // num_heads) * cache.size(1)
    
    print(f"\nFinal comparison:")
    print(f"  MLA cache: {total_cache} values")
    print(f"  Full KV cache would be: {full_kv_cache} values")
    print(f"  Savings: {full_kv_cache / total_cache:.1f}x")


def demo_conversion():
    """Demonstrate MHA to MLA conversion."""
    print("\n" + "="*80)
    print("DEMO 3: Converting MHA to MLA")
    print("="*80)
    
    embed_dim = 128
    num_heads = 8
    latent_dim = 32
    
    # Create and "train" MHA
    mha = StandardMultiHeadAttention(embed_dim, num_heads)
    
    print(f"\nOriginal MHA:")
    mha_params = MLABenchmark.count_parameters(mha)
    print(f"  Parameters: {mha_params:,}")
    
    # Convert to MLA
    mla = convert_mha_to_mla_svd(mha, latent_dim)
    
    print(f"\nConverted MLA:")
    mla_params = MLABenchmark.count_parameters(mla)
    print(f"  Parameters: {mla_params:,}")
    print(f"  Latent dim: {latent_dim}")
    
    # Test
    x = torch.randn(2, 10, embed_dim)
    
    with torch.no_grad():
        mha_out = mha(x)
        mla_out, _ = mla(x)
        diff = (mha_out - mla_out).abs().mean()
    
    print(f"\nOutput difference: {diff:.6f}")
    print(f"  (Small difference expected due to low-rank approximation)")


def demo_memory_comparison():
    """Compare memory usage across configurations."""
    print("\n" + "="*80)
    print("DEMO 4: Memory Comparison")
    print("="*80)
    
    configs = [
        (8192, 4096, 64, 4096, "MHA (full)"),
        (8192, 4096, 64, 1024, "MLA (d_c=1024)"),
        (8192, 4096, 64, 512, "MLA (d_c=512)"),
        (8192, 4096, 64, 256, "MLA (d_c=256)"),
    ]
    
    print(f"\nKV Cache Size Comparison (32 layers):")
    print(f"{'Config':<20} {'Cache (GB)':<12} {'Reduction':<12}")
    print("-" * 50)
    
    for seq_len, embed_dim, num_heads, latent_dim, name in configs:
        result = MLABenchmark.calculate_kv_cache_size(
            seq_len, embed_dim, num_heads, latent_dim, num_layers=32
        )
        
        if "full" in name:
            cache_gb = result['mha_cache_mb'] / 1024
            reduction = "-"
        else:
            cache_gb = result['mla_cache_mb'] / 1024
            reduction = f"{result['reduction_factor']:.1f}x"
        
        print(f"{name:<20} {cache_gb:>10.2f}  {reduction:>10}")


def demo_speed_comparison():
    """Compare inference speed."""
    print("\n" + "="*80)
    print("DEMO 5: Speed Comparison")
    print("="*80)
    
    configs = [
        (4, 512, 256, 16, 256, "No compression"),
        (4, 512, 256, 16, 64, "4x compression"),
        (4, 512, 256, 16, 32, "8x compression"),
    ]
    
    print(f"\nInference Speed:")
    print(f"{'Config':<20} {'MHA (ms)':<12} {'MLA (ms)':<12} {'Relative':<12}")
    print("-" * 60)
    
    for batch, seq, dim, heads, latent, name in configs:
        result = MLABenchmark.benchmark_speed(
            batch, seq, dim, heads, latent, num_iterations=50
        )
        
        print(f"{name:<20} {result['mha_time_ms']:>10.2f}  "
              f"{result['mla_time_ms']:>10.2f}  {result['relative_speed']:>10.2f}x")


# ============================================================================
# 7. VISUALIZATION
# ============================================================================

def visualize_compression():
    """Visualize the compression effect."""
    import matplotlib.pyplot as plt
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Standard MHA
    num_heads = 16
    head_dim = 16
    
    ax1.set_title('Standard Multi-Head Attention', fontsize=14, fontweight='bold')
    for i in range(num_heads):
        ax1.add_patch(plt.Rectangle((i, 0), 0.8, 0.8, facecolor='blue', alpha=0.6))
        ax1.text(i+0.4, 0.4, f'K{i}', ha='center', va='center', fontsize=8)
        ax1.add_patch(plt.Rectangle((i, 1), 0.8, 0.8, facecolor='red', alpha=0.6))
        ax1.text(i+0.4, 1.4, f'V{i}', ha='center', va='center', fontsize=8)
    
    ax1.set_xlim(-1, num_heads)
    ax1.set_ylim(-0.5, 2.5)
    ax1.set_aspect('equal')
    ax1.axis('off')
    ax1.text(num_heads/2, -0.3, f'Total: {num_heads * 2} matrices', 
            ha='center', fontsize=12, fontweight='bold')
    
    # MLA
    latent_dim = 4
    
    ax2.set_title('Multi-Head Latent Attention', fontsize=14, fontweight='bold')
    
    # Compressed representation
    for i in range(latent_dim):
        ax2.add_patch(plt.Rectangle((i, 0), 0.8, 0.8, facecolor='green', alpha=0.8))
        ax2.text(i+0.4, 0.4, f'C{i}', ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Arrows to heads
    for i in range(num_heads):
        x_start = latent_dim / 2
        x_end = i
        ax2.annotate('', xy=(x_end+0.4, 1.5), xytext=(x_start, 0.9),
                    arrowprops=dict(arrowstyle='->', alpha=0.3, color='gray'))
    
    # Decompressed heads (shown smaller)
    for i in range(0, num_heads, 4):  # Show every 4th for clarity
        ax2.add_patch(plt.Rectangle((i, 1.5), 0.8, 0.4, facecolor='blue', alpha=0.4))
        ax2.text(i+0.4, 1.7, f'K{i}', ha='center', va='center', fontsize=6)
    
    ax2.set_xlim(-1, num_heads)
    ax2.set_ylim(-0.5, 2.5)
    ax2.set_aspect('equal')
    ax2.axis('off')
    ax2.text(num_heads/2, -0.3, 
            f'Compressed: {latent_dim} dims\n({(num_heads * 2) / latent_dim:.0f}x reduction)', 
            ha='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('/tmp/mla_compression.png', dpi=150, bbox_inches='tight')
    print("\nVisualization saved to /tmp/mla_compression.png")
    plt.close()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("MULTI-HEAD LATENT ATTENTION: COMPLETE IMPLEMENTATION")
    print("="*80)
    
    torch.manual_seed(42)
    
    # Run all demos
    demo_basic_mla()
    demo_kv_cache()
    demo_conversion()
    demo_memory_comparison()
    demo_speed_comparison()
    
    # Visualization
    visualize_compression()
    
    print("\n" + "="*80)
    print("All demonstrations completed successfully!")
    print("="*80)


# ============================================================================
# ADDITIONAL UTILITIES
# ============================================================================

def analyze_singular_values(weight_matrix):
    """
    Analyze singular value distribution to determine optimal latent dim.
    
    Args:
        weight_matrix: Weight matrix to analyze
    
    Returns:
        dict with singular value statistics
    """
    U, S, Vt = torch.linalg.svd(weight_matrix, full_matrices=False)
    
    # Cumulative explained variance
    total_energy = (S ** 2).sum()
    cumulative_energy = torch.cumsum(S ** 2, dim=0) / total_energy
    
    # Find dimension for 95%, 99% energy
    dim_95 = (cumulative_energy > 0.95).nonzero()[0].item() + 1
    dim_99 = (cumulative_energy > 0.99).nonzero()[0].item() + 1
    
    return {
        'singular_values': S.numpy(),
        'dim_for_95_energy': dim_95,
        'dim_for_99_energy': dim_99,
        'effective_rank': (S > 0.01 * S[0]).sum().item()
    }


if __name__ == "__main__":
    # Example: Analyze a random weight matrix
    W = torch.randn(4096, 4096)
    stats = analyze_singular_values(W)
    print(f"\nSingular value analysis:")
    print(f"  95% energy captured by top {stats['dim_for_95_energy']} dims")
    print(f"  99% energy captured by top {stats['dim_for_99_energy']} dims")
    print(f"  Effective rank: {stats['effective_rank']}")

In [ ]:
# Create MLA layer
mla = MultiHeadLatentAttention(
    embed_dim=512,
    num_heads=32,
    latent_dim=128  # 4× compression
)

# Forward pass
x = torch.randn(16, 100, 512)
output, _ = mla(x)

With KV Caching

In [ ]:
mla = CachedMultiHeadLatentAttention(
    embed_dim=512,
    num_heads=32,
    latent_dim=128
)

# Generation
cache = None
for step in range(seq_len):
    x_i = ...
    output, cache = mla(x_i, past_c_kv=cache, use_cache=True)

 Convert Existing Model

In [ ]:
# Load pretrained MHA
mha = StandardMultiHeadAttention(embed_dim=768, num_heads=12)
mha.load_state_dict(checkpoint)

# Convert to MLA
mla = convert_mha_to_mla_svd(mha, latent_dim=192)

# Fine-tune if needed

 Compression Strategy

In [ ]:
# Compress once
c_kv = self.kv_compress(x)  # [batch, seq, latent_dim]

# Decompress per head
for i in range(num_heads):
    k_i = self.k_decompress[i](c_kv)  # [batch, seq, head_dim]

Cache Only Latent

In [ ]:
# Store compressed representation
cache = c_kv  # [batch, seq, latent_dim]

# NOT full K, V for each head!
# NOT [batch, num_heads, seq, head_dim]

 SVD Initialization

In [ ]:
# For stable conversion
U, S, Vt = torch.linalg.svd(W)
W_down = U[:, :d_c] @ sqrt(S[:d_c])
W_up = sqrt(S[:d_c]) @ Vt[:d_c, :]